![MOMENTUM · Hackatón de DRL · Entrenamiento](img/banner-train.jpg)

# **Código para el entrenamiento del agente**

Este cuaderno contiene el código necesario para entrenar un agente de [Stable Baselines 3](https://stable-baselines3.readthedocs.io/en/master/) en un entorno de la colección [`highway-env`](https://highway-env.farama.org/).

Primero, importamos las librerías necesarias:

In [ ]:
import gymnasium as gym
import highway_env

from stable_baselines3 import A2C, DQN, HER, PPO
from matplotlib import pyplot as plt
%matplotlib inline

Dada la componente aleatoria del entrenamiento, fijamos una semilla para el agente y el sampler de Optuna. Esto permite repetir el experimento bajo las mismas condiciones. Aún así, no garantiza resultados idénticos entre distintas versiones de las librerías o hardware.

In [ ]:
import random

# Fijar la semilla
seed = 42
random.seed(seed)

## **1. Entorno de entrenamiento**

### **1.1 Definición del entorno**

**Dado que el entrenamiento de los agentes puede llevar bastante tiempo, se recomienda utilizar una variante simplificada llamada [`highway-fast-v0`](https://highway-env.farama.org/environments/highway/#faster-variant)**, que permite entrenar hasta 15 veces más rápido. No obstante, la evaluación será en el entorno no simplificado (`highway-v0`). Si bien es cierto que **un agente entrenado en `highway-fast-v0` puede generalizar y comportarse bien en `highway-v0`**, se recomienda usar el entorno simplificado para los entrenamientos más costosos (probar distintos algoritmos, ajustar hiperparámetros) y, una vez encontrada una buena configuración para el entrenamiento, entrenar el modelo definitivo en el entorno no simplificado, siempre y cuando sea posible por cuestiones de tiempo.

In [ ]:
# Indica el entorno para el entrenamiento. Puedes cambiarlo por otro entorno de highway-env si lo deseas.
TRAIN_ENV_NAME = 'highway-fast-v0'

Dicho esto, visualizamos una instancia del entorno donde se va a entrenar el agente.

In [ ]:
# Crear el entorno
env = gym.make(TRAIN_ENV_NAME, render_mode='rgb_array')

# Fijar una semilla para reproducibilidad
env.reset(seed=seed)

# Mostrar el estado inicial
plt.imshow(env.render())
plt.show()

A continuación, se muestran el espacio de observaciones y el espacio de acciones por defecto.

In [ ]:
# Espacio de observaciones
print("Espacio de observaciones:", env.observation_space)

# Espacio de acciones
print("Espacio de acciones:", env.action_space)

### **1.2 Definición y aplicación de wrappers**

Una vez creado el entorno, definimos y aplicamos los wrappers que consideremos necesarios. **Es importante tener en cuenta en qué orden se aplican, y mantener el mismo orden en la evaluación**. Recuerda que puedes consultar la documentación de los wrappers [aquí](https://gymnasium.farama.org/api/wrappers).

In [ ]:
# Define tus wrappers aqui

# class wrapper_1(gym.ObservationWrapper):
# ...

# class wrapper_2(gym.ActionWrapper):
# ...


# Centraliza aqui la configuracion y los wrappers de todos los entornos.
def make_env():
    new_env = gym.make(TRAIN_ENV_NAME, render_mode='rgb_array')
    # new_env = wrapper_1(new_env)
    # new_env = wrapper_2(new_env)
    return new_env


***

## **2. Entrenamiento de los agentes**

Una vez definido el entorno y los wrappers, especificaremos la configuración para el entrenamiento de los agentes: número de episodios de entrenamiento, evaluación, combinaciones de hiperparámetros a probar, etc.

**<span style="color:red">¡Atención!</span> El código de las siguientes celdas guarda el mejor agente resultante en un archivo dentro de la carpeta `agentes/<EXPERIMENT_NAME>`. Si ya existe una carpeta con el mismo nombre, el agente resultante de la última ejecución sobreescribirá al anterior, así que recuerda cambiar el valor de la variable `EXPERIMENT_NAME` y ejecutar la celda cada vez que modifiques la configuración del entrenamiento.**

In [ ]:
EXPERIMENT_NAME = "DQN_10_episodes" # Nombre del experimento para guardar resultados

**Las variables definidas en la siguiente celda no se deben modificar**:

In [ ]:
# Variables fijas que NO deben ser modificadas
N_JOBS = 1 # Numero de trabajos en paralelo para Optuna
TIMESTEPS_PER_EPISODE = 40 # Numero de pasos por episodio
EVAL_EPISODES = 5 # Numero de episodios que dura cada evaluacion

**Las siguientes variables sí deben ser ajustadas**:

In [ ]:
# Variables que SI pueden ser modificadas
TRAIN_EPISODES = 10 # Numero de episodios de entrenamiento
N_COMBINACIONES = 2 # Numero de combinaciones a explorar durante la optimizacion de hiperparametros

### **2.1 Definición de callbacks personalizados**

En la siguiente celda **definimos las clases** de los callbacks personalizados. Las **instancias** se crearán después, dentro de `objective`, para que cada entrenamiento tenga sus propios callbacks.

Para más información sobre callbacks personalizados, consulta [la documentación](https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html).

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback


# Define aqui las clases de tus callbacks personalizados.
# No olvides instanciarlos mas adelante en la funcion objective.

# class MiCallback(BaseCallback):
#     ...


### **2.2 Definición del entrenamiento y de los hiperparámetros**

Definimos la función `objective(trial)`, que Optuna ejecutará una vez por cada combinación de hiperparámetros. En ella seleccionamos los hiperparámetros, creamos el agente e instanciamos los callbacks definidos en la celda anterior. Ten en cuenta que esta celda solo define la función; el estudio de Optuna se ejecuta en la siguiente celda.


In [ ]:
import optuna
import numpy as np
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import CallbackList, StopTrainingOnMaxEpisodes


# Semillas de validacion comunes a todos los trials, distintas de entrenamiento.
EVAL_SEEDS = [seed + 1000 + i for i in range(EVAL_EPISODES)]


# Definir la funcion para el/los entrenamiento/s
def objective(trial):

    # Hiperparametros a optimizar
    learning_rate = trial.suggest_float(
        "learning_rate", 5e-4, 5e-4, log=True
    )

    # Resto de hiperparametros a optimizar
    # ...


    train_env = None
    eval_env = None
    try:
        train_env = Monitor(make_env())
        eval_env = DummyVecEnv([lambda: Monitor(make_env())])

        # Crear agente
        agent = DQN(
            "MlpPolicy",
            train_env,
            seed=seed,
            learning_rate=learning_rate,
            # Resto de hiperparametros
            # ...
            verbose=0,
        )


        # Instanciar los callbacks para este trial
        stop_training_callback = StopTrainingOnMaxEpisodes(max_episodes=TRAIN_EPISODES, verbose=1) # Limita el numero de episodios al indicado

        # Instancia aqui las clases definidas en la celda de callbacks
        # custom_callback = MiCallback(verbose=1)


        # Agrupar las instancias que se usaran en agent.learn
        callbacks = CallbackList([
            stop_training_callback,
            # custom_callback,
        ])


        # Entrenamiento
        agent.learn(total_timesteps=TRAIN_EPISODES * TIMESTEPS_PER_EPISODE,
                    callback=callbacks,
                    )


        # Cada episodio empieza con la misma semilla en todos los trials.
        # VecEnv.seed aplica la semilla en el reset que hace evaluate_policy.
        episode_rewards = []
        for eval_seed in EVAL_SEEDS:
            eval_env.seed(eval_seed)
            rewards, _ = evaluate_policy(
                agent,
                eval_env,
                n_eval_episodes=1,
                deterministic=True,
                return_episode_rewards=True,
            )
            episode_rewards.extend(rewards)
        mean_reward = float(np.mean(episode_rewards))
        std_reward = float(np.std(episode_rewards))

        # Guardar el agente si es el mejor hasta ahora
        completed_trials = trial.study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))

        if not completed_trials or mean_reward > trial.study.best_value:
            agent.save(f"agentes/{EXPERIMENT_NAME}/best_model.zip")


        return mean_reward
    finally:
        if eval_env is not None:
            eval_env.close()
        if train_env is not None:
            train_env.close()


### **2.3 Ejecución del entrenamiento**

Una vez ejecutadas las celdas de definición de callbacks y de `objective`, creamos el estudio de Optuna y lanzamos los entrenamientos. Cada llamada a `objective` crea un agente y nuevas instancias de sus callbacks.

Si modificas una clase de callback, vuelve a ejecutar su celda antes de lanzar de nuevo el estudio. Si cambias los parámetros de sus instancias o la lista de callbacks, vuelve a ejecutar también la celda de `objective`.


In [ ]:
import time

# Crear estudio de Optuna
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=seed),
)

# Guardar la hora de inicio
start_time = time.time()

# Ejecutar entrenamiento
study.optimize(objective, n_trials=N_COMBINACIONES, n_jobs=N_JOBS)

# Guardar la hora de finalizacion
end_time = time.time()

 Una vez finalizado el entrenamiento, se muestra su duración, la mejor combinación de hiperparámetros explorada y la recompensa obtenida con dicha combinación. Esta información también se escribe en el archivo `agentes/<EXPERIMENT_NAME>/info_train.txt`.

In [ ]:
# Mostrar informacion sobre el entrenamiento
print(f"El entrenamiento {EXPERIMENT_NAME} ha durado {end_time - start_time:.2f} segundos.")
print(f"Mejor recompensa obtenida: {study.best_value:.2f}")
print(f"Se han probado {N_COMBINACIONES} combinaciones de hiperparametros.")
print(f"Mejores hiperparametros:\n{study.best_params}")


# Escribir informacion util sobre el entrenamiento en un archivo de texto
with open(f"./agentes/{EXPERIMENT_NAME}/info_train.txt", "w") as f:
    f.write(f"Informacion sobre el experimento {EXPERIMENT_NAME}\n")
    f.write(f"\n - Duracion: {end_time - start_time:.2f} segundos.")
    f.write(f"\n - La mejor recompensa obtenida ha sido: {study.best_value:.2f}")
    f.write(f"\n - Se han probado {N_COMBINACIONES} combinaciones de hiperparametros.")
    f.write(f"\n - Los hiperparametros del mejor modelo encontrado son:\n\t{study.best_params}")


Estos resultados se obtienen durante el entrenamiento. Sin embargo, **para medir el desempeño del mejor agente de forma justa, se debe evaluar en un entorno diferente**. Para ello, utiliza el **[notebook de test](test.ipynb)**. En él, encontrarás el código necesario para evaluar un agente ya entrenado, además de la obtención de métricas que facilitan el análisis de la política aprendida.